# Comparative Specifications — DiD Analysis

**Standalone notebook** — loads pre-processed data from `analysis_data.parquet`.
Run `preprocess.py` once to generate the parquet from the two Excel source files.

## Outline
| Section | Specification | Key coefficient |
|---|---|---|
| 0 | Data loading & sample construction | — |
| 1 | Simple OLS | `treat` |
| 2 | Difference-in-Differences | `treat:post` |
| 3 | Triple Difference (DDD) | `treat:post:impl` |
| 4 | Overview table | all specs side-by-side |
| 5 | Robustness checks | various |
| 6 | Extension: resource-efficiency domains | `treat` / `treat:post` / `treat:post:impl` |

## 0. Data Loading & Feature Engineering

### 0.1 Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

BASE = Path('.').resolve()  # folder containing the data files
print(f"Working directory: {BASE}")

Working directory: C:\Users\zolb\Files C\ClaudeFolder\EcoLink


### 0.2 Load data

In [2]:
df = pd.read_parquet(BASE / 'analysis_data.parquet')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'  years:      {sorted(df["year"].unique())}')
print(f'  countries:  {df["ipscntry"].nunique()}')
print(f'  treat=1:    {(df["treat"]==1).sum():,}  |  treat=0: {(df["treat"]==0).sum():,}')
print(f'  impl=1 cntrs: {df.loc[df["implementation_country"]==1,"ipscntry"].nunique()}')
print(f'  impl=0 cntrs: {df.loc[df["implementation_country"]==0,"ipscntry"].nunique()}')

Loaded: 28,263 rows x 46 cols
  years:      [np.int64(2021), np.int64(2024)]
  countries:  27
  treat=1:    4,236  |  treat=0: 24,000
  impl=1 cntrs: 20
  impl=0 cntrs: 7


### 0.3 Treatment variables
`treat = 1` if firm has ≥ 250 employees **or** turnover ≥ €10 M.
Pre-computed by `preprocess.py`; confirmed below.

In [3]:
# Treatment variables loaded from parquet — verify key columns
print('treat distribution:')
print(df['treat'].value_counts(dropna=False).to_string())
print('post distribution:')
print(df['post'].value_counts().to_string())

treat distribution:
treat
0.0    24000
1.0     4236
NaN       27
post distribution:
post
0    14215
1    14048


### 0.4 Outcome variables
Pre-computed by `preprocess.py`.

- **`target_engaged`**: 1 if firm has or plans a climate strategy (from q14)
- **`firm_growth_ord`**: ordinal −1/0/1 (decreased / unchanged / increased)
- **`firm_growth_ord_d`**: binary 1 = increased, 0 = flat or declined

In [4]:
# Outcome variables loaded from parquet — verify means
for col in ['target_engaged', 'firm_growth_ord', 'firm_growth_ord_d']:
    print(f'  {col}: mean={df[col].mean():.3f}  N={df[col].notna().sum():,}')

  target_engaged: mean=0.508  N=28,263
  firm_growth_ord: mean=0.231  N=28,263
  firm_growth_ord_d: mean=0.420  N=28,263


### 0.5 Firm-level controls
Pre-computed by `preprocess.py`. Available columns:

| Variable | Description |
|---|---|
| `firm_age_ord` | Ordinal 1–4 (1=youngest) |
| `firm_age_ord_d` | Binary: 1 if founded before 2016/2014 |
| `east_west` | 1=Western EU, 0=Eastern EU |
| `*_maturity_d` | 10 resource-efficiency domain action dummies |
| `green_staff_any` | 1 if any green-dedicated staff |
| `investment_dummy` | 1 if any green investment |
| `barrier_*` / `barrier_*_d` | Institutional / capability / market barriers |
| `support_*` | Knowledge, financial, external, internal support |
| `green_offer_ord_d` / `green_turnover_pct_d` | Green product/revenue dummies |

In [5]:
# Controls loaded from parquet — spot-check
ctrl_cols = ['firm_age_ord','east_west','green_staff_any','investment_dummy',
             'barrier_institutional_d','support_knowledge_d','green_offer_ord_d']
print(df[ctrl_cols].describe().round(3).to_string())

       firm_age_ord  east_west  green_staff_any  investment_dummy  barrier_institutional_d  support_knowledge_d  green_offer_ord_d
count     28263.000  28263.000        28263.000         28263.000                28263.000            28263.000          28263.000
mean          3.695      0.580            0.486             0.825                    0.492                0.596              0.466
std           0.688      0.493            0.500             0.380                    0.500                0.491              0.499
min           1.000      0.000            0.000             0.000                    0.000                0.000              0.000
25%           4.000      0.000            0.000             1.000                    0.000                0.000              0.000
50%           4.000      1.000            0.000             1.000                    0.000                1.000              0.000
75%           4.000      1.000            1.000             1.000                  

### 0.6 Analysis samples

In [6]:
# Main sample: implementing countries (impl=1), complete cases
df_main  = df[df['sample_main'] == 1]
df_clean = df_main.dropna(subset=[
    "target_engaged", "treat", "post", "firm_age_ord", "nace_b", "ipscntry"
]).copy()

# DDD sample: all countries (impl=0 + impl=1)
df_ddd = df.dropna(subset=[
    "target_engaged", "treat", "post", "firm_age_ord", "nace_b",
    "ipscntry", "implementation_country"
]).copy()
df_ddd["impl"] = df_ddd["implementation_country"]

# Working samples (drop residual NAs on firm_growth_ord_d)
df_s = df_clean.dropna(subset=['firm_growth_ord_d']).copy()  # OLS / DiD
df_d = df_ddd.dropna(subset=['firm_growth_ord_d']).copy()    # DDD

# Extended FE dummies for Section 4
df_d["cntry_treat"] = df_d["ipscntry"].astype(str) + "_" + df_d["treat"].astype(str)
df_d["cntry_post"]  = df_d["ipscntry"].astype(str) + "_" + df_d["post"].astype(str)

print(f"OLS/DiD sample (impl=1 only): {df_s.shape[0]:,} obs")
print(f"DDD sample (all countries):   {df_d.shape[0]:,} obs")
print(f"  impl=1 (policy countries):  {(df_d['impl']==1).sum():,}")
print(f"  impl=0 (never-treated):     {(df_d['impl']==0).sum():,}")

def cl(df_):
    """Clustered SE by country."""
    return {"cov_type": "cluster", "cov_kwds": {"groups": df_["ipscntry"]}}

OLS/DiD sample (impl=1 only): 21,396 obs
DDD sample (all countries):   28,236 obs
  impl=1 (policy countries):  21,396
  impl=0 (never-treated):     6,840


---
## 1. Simple OLS
Cross-sectional estimate of the firm-size (`treat`) effect with no pre/post
variation exploited. Two outcomes × two FE choices = 4 models.

| Model | Outcome | Country FE |
|---|---|---|
| 1a | `target_engaged` | No |
| 1b | `target_engaged` | Yes (`C(ipscntry)`) |
| 1c | `firm_growth_ord_d` | No |
| 1d | `firm_growth_ord_d` | Yes (`C(ipscntry)`) |

In [7]:
s1a = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1b = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s1c = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1d = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("1a  target_engaged    no FE",      s1a),
               ("1b  target_engaged    country FE",  s1b),
               ("1c  firm_growth_ord_d no FE",       s1c),
               ("1d  firm_growth_ord_d country FE",  s1d)]:
    print(f"{lbl}: treat = {m.params['treat']:.4f}  (p={m.pvalues['treat']:.3f})")

1a  target_engaged    no FE: treat = 0.2608  (p=0.000)
1b  target_engaged    country FE: treat = 0.2302  (p=0.000)
1c  firm_growth_ord_d no FE: treat = 0.1696  (p=0.000)
1d  firm_growth_ord_d country FE: treat = 0.1626  (p=0.000)


---
## 2. Difference-in-Differences (DiD)
`treat:post` is the DiD coefficient — the additional change for large firms
after the policy (`post=1`, year 2024) relative to small firms.  
Sample restricted to implementing countries (`impl=1`).

| Model | Outcome | Country FE |
|---|---|---|
| 2a | `target_engaged` | No |
| 2b | `target_engaged` | Yes |
| 2c | `firm_growth_ord_d` | No |
| 2d | `firm_growth_ord_d` | Yes |

In [8]:
s2a = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2b = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s2c = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2d = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("2a  target_engaged    no FE",      s2a),
               ("2b  target_engaged    country FE",  s2b),
               ("2c  firm_growth_ord_d no FE",       s2c),
               ("2d  firm_growth_ord_d country FE",  s2d)]:
    print(f"{lbl}: treat:post = {m.params['treat:post']:.4f}  (p={m.pvalues['treat:post']:.3f})")

2a  target_engaged    no FE: treat:post = 0.0634  (p=0.028)
2b  target_engaged    country FE: treat:post = 0.0649  (p=0.014)
2c  firm_growth_ord_d no FE: treat:post = -0.1062  (p=0.005)
2d  firm_growth_ord_d country FE: treat:post = -0.1004  (p=0.007)


---
## 3. Difference-in-Difference-in-Differences (DDD)
Adds `impl` (= `implementation_country`) as a third dimension.  
Countries with `impl=0` are **never-treated controls**, removing global time
trends that affect large firms everywhere regardless of policy.  
`treat:post:impl` is the DDD coefficient.

> **Note:** `C(ipscntry)` cannot be included — `impl` is constant within
> countries and is perfectly collinear with country FEs. Sector + age FEs
> are used for the 'with FE' variant.

| Model | Outcome | Controls |
|---|---|---|
| 3a | `target_engaged` | Sector only |
| 3b | `target_engaged` | Sector + firm age |
| 3c | `firm_growth_ord_d` | Sector only |
| 3d | `firm_growth_ord_d` | Sector + firm age |

In [9]:
s3a = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3b = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))
s3c = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3d = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))

for lbl, m in [("3a  target_engaged    sector FE",      s3a),
               ("3b  target_engaged    sector+age FE",  s3b),
               ("3c  firm_growth_ord_d sector FE",      s3c),
               ("3d  firm_growth_ord_d sector+age FE",  s3d)]:
    print(f"{lbl}: treat:post:impl = {m.params['treat:post:impl']:.4f}  "
          f"(p={m.pvalues['treat:post:impl']:.3f})")

3a  target_engaged    sector FE: treat:post:impl = 0.1749  (p=0.001)
3b  target_engaged    sector+age FE: treat:post:impl = 0.1754  (p=0.001)
3c  firm_growth_ord_d sector FE: treat:post:impl = -0.0520  (p=0.427)
3d  firm_growth_ord_d sector+age FE: treat:post:impl = -0.0524  (p=0.430)


---
## 4. Overview Table

All parametric specifications (Sections 1–3) side-by-side.
Standard errors clustered by country (`ipscntry`).
Significance: \* p<0.1, \*\* p<0.05, \*\*\* p<0.01.

| Section | Models | Key coefficient |
|---|---|---|
| 1 OLS  | 1a–1d | `treat` |
| 2 DiD  | 2a–2d | `treat:post` |
| 3 DDD  | 3a–3d | `treat:post:impl` |

In [10]:
from statsmodels.iolib.summary2 import summary_col

comp_table = summary_col(
    [s1a, s1b, s1c, s1d, s2a, s2b, s2c, s2d, s3a, s3b, s3c, s3d],
    model_names=[
        "(1a)","(1b)","(1c)","(1d)",
        "(2a)","(2b)","(2c)","(2d)",
        "(3a)","(3b)","(3c)","(3d)",
    ],
    stars=True,
    float_format="%0.4f",
    regressor_order=["treat", "treat:post", "treat:post:impl"],
    drop_omitted=True,
    info_dict={
        "N":         lambda x: f"{int(x.nobs)}",
        "R-squared": lambda x: f"{x.rsquared:.3f}",
        "Outcome":   lambda x: x.model.endog_names,
    },
)
print(comp_table)


                     (1a)           (1b)             (1c)              (1d)            (2a)           (2b)             (2c)              (2d)            (3a)           (3b)             (3c)              (3d)      
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
treat           0.2608***      0.2302***      0.1696***         0.1626***         0.2264***      0.1952***      0.2247***         0.2141***         0.1939***      0.1942***      0.1025**          0.1082**         
                (0.0323)       (0.0252)       (0.0195)          (0.0146)          (0.0249)       (0.0184)       (0.0239)          (0.0239)          (0.0397)       (0.0399)       (0.0504)          (0.0515)         
treat:post                                                                        0.0634**       0.0649**       -0.1062***        -0.1004***   

In [11]:
with open('overview_table.txt', 'w', encoding='utf-8') as f:
    f.write(str(comp_table))
print('Saved: overview_table.txt')

Saved: overview_table.txt


---
## 5. Robustness Checks

| Subsection | Check |
|---|---|
| 5.1 | Logistic regression for binary growth outcome |
| 5.2 | Repeated cross-section composition checks |
| 5.3 | Placebo treatment definitions |
| 5.4 | DiD and DDD with alternative outcome variables |
| 5.5 | OLS / DiD / DDD with East\u2013West control |

### 5.1  Logistic Regression for Binary Growth Outcome

`firm_growth_ord_d` is binary (0/1). The OLS linear probability model (LPM) used in
Sections 1\u20133 is consistent but may predict probabilities outside [0, 1].  These
logit models replicate the OLS, DiD, and DDD structures.
Coefficients are log-odds; the adjacent LPM column (s1b/s2b/s3d) enables comparison.

In [12]:
# 5.1  Logit models for firm_growth_ord_d
lg_ols = smf.logit('firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)',
                    data=df_s).fit(**cl(df_s))
lg_did = smf.logit('firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)',
                    data=df_s).fit(**cl(df_s))
lg_ddd = smf.logit('firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)',
                    data=df_d).fit(**cl(df_d))

logit_specs = [
    ('OLS analog', lg_ols, 'treat',           s1b, 'treat'),
    ('DiD',        lg_did, 'treat:post',       s2b, 'treat:post'),
    ('DDD',        lg_ddd, 'treat:post:impl',  s3d, 'treat:post:impl'),
]
print('%-14s %11s %8s %8s   %11s %8s %8s' % (
      'Model', 'Logit coef', 'SE', 'p', 'LPM coef', 'SE', 'p'))
print('-' * 76)
for lbl, ml, kl, mo, ko in logit_specs:
    cl_ = ml.params[kl]; sl_ = ml.bse[kl]; pl_ = ml.pvalues[kl]
    co_ = mo.params[ko]; so_ = mo.bse[ko]; po_ = mo.pvalues[ko]
    sg  = '***' if pl_<0.01 else '**' if pl_<0.05 else '*' if pl_<0.10 else ''
    print('%-14s %+11.4f %8.4f %6.3f%-3s  %+11.4f %8.4f %6.3f' % (
          lbl, cl_, sl_, pl_, sg, co_, so_, po_))
print()
print('Logit coefs = log-odds. LPM coefs from s1b/s2b/s3d. SE clustered by ipscntry.')

Optimization terminated successfully.
         Current function value: 0.674121
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.672535
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.671301
         Iterations 4
Model           Logit coef       SE        p      LPM coef       SE        p
----------------------------------------------------------------------------
OLS analog         +0.6880   0.0788  0.000***      +0.2302   0.0252  0.000
DiD                -0.4385   0.1549  0.005***      +0.0649   0.0265  0.014
DDD                -0.2165   0.2673  0.418         -0.0524   0.0664  0.430

Logit coefs = log-odds. LPM coefs from s1b/s2b/s3d. SE clustered by ipscntry.


### 5.2  Repeated Cross-Section Composition Checks

The survey is not a true firm panel: each wave samples independently.
Observed outcome changes could reflect shifts in sample composition (sector mix,
size distribution, geography) rather than genuine firm-level responses.
We test whether key observable characteristics differ across the 2022 and 2024 waves.

In [13]:
# 5.2  Composition checks
from scipy.stats import chi2_contingency

print('Chi-square tests: independence of composition variable and survey year (df_s)')
print('%-30s %8s %4s %8s' % ('Variable', 'chi2', 'df', 'p-value'))
print('-' * 55)
for col in ['treat', 'nace_b', 'east_west', 'firm_age_ord']:
    sub = df_s[['year', col]].dropna()
    ch, pv, dof, _ = chi2_contingency(pd.crosstab(sub['year'], sub[col]))
    sg = '***' if pv<0.01 else '**' if pv<0.05 else '*' if pv<0.10 else ''
    print('%-30s %8.2f %4d %8.3f%s' % (col, ch, dof, pv, sg))

print()
print('Mean outcomes by year x treat group (df_s)')
tbl = (df_s.groupby(['year', 'treat'])
           [['target_engaged', 'firm_growth_ord_d', 'firm_age_ord']]
           .mean())
print(tbl.to_string(float_format='%.4f'))

print()
print('Sample size by year (df_s)')
print(df_s.groupby('year').size().rename('n').to_string())

Chi-square tests: independence of composition variable and survey year (df_s)
Variable                           chi2   df  p-value
-------------------------------------------------------
treat                             48.72    1    0.000***
nace_b                            31.28    3    0.000***
east_west                          0.62    1    0.433
firm_age_ord                      44.47    3    0.000***

Mean outcomes by year x treat group (df_s)
            target_engaged  firm_growth_ord_d  firm_age_ord
year treat                                                 
2021 0.0            0.4606             0.3800        3.6753
     1.0            0.6898             0.5899        3.9237
2024 0.0            0.4481             0.4339        3.6876
     1.0            0.7412             0.5406        3.8852

Sample size by year (df_s)
year
2021    10772
2024    10624


### 5.3  Placebo Treatment Definitions

The baseline `treat` indicator is 1 when a firm meets **either** the employee
(\u2265 250) **or** turnover (\u2265 \u20ac10 M) threshold (CSRD Article 3).
We re-estimate the DiD and DDD using three alternative definitions built from the
same pre-computed binary indicators in the parquet:

| Definition | Rule |
|---|---|
| Baseline (any) | employee \u2265 250 **OR** turnover \u2265 \u20ac10 M (main spec) |
| Employees only | `employee_treatment = 1` regardless of turnover |
| Turnover only  | `turnover_treatment = 1` regardless of employees |
| Both (strict)  | `employee_treatment = 1` **AND** `turnover_treatment = 1` |

In [14]:
# 5.3  Placebo treatment definitions
df_s_pl = df_s.copy()
df_d_pl = df_d.copy()
for df_ in [df_s_pl, df_d_pl]:
    df_['treat_emp']    = df_['employee_treatment'].astype(float)
    df_['treat_turn']   = df_['turnover_treatment'].astype(float)
    df_['treat_strict'] = (df_['employee_treatment'] * df_['turnover_treatment']).astype(float)

print('Treatment group size by definition (df_s)')
for tcol, lbl in [('treat','Baseline'),('treat_emp','Employees only'),
                   ('treat_turn','Turnover only'),('treat_strict','Both (strict)')]:
    n = int(df_s_pl[tcol].fillna(0).sum())
    print('  %-18s treat=1: %5d  (%.1f%%)' % (lbl, n, 100*n/len(df_s_pl)))

print()
print('%-20s %10s %8s %8s   %10s %8s %8s' % (
      'Treatment def.', 'DiD coef', 'SE', 'p', 'DDD coef', 'SE', 'p'))
print('-' * 80)
for label, tcol in [('Baseline (any)',  'treat'),
                     ('Employees only', 'treat_emp'),
                     ('Turnover only',  'treat_turn'),
                     ('Both (strict)',  'treat_strict')]:
    if tcol == 'treat':
        md_ = s2a;  m3_ = s3b
    else:
        ss = df_s_pl.dropna(subset=[tcol])
        sd = df_d_pl.dropna(subset=[tcol])
        md_ = smf.ols('target_engaged ~ %s*post + C(nace_b) + C(firm_age_ord)' % tcol,
                       data=ss).fit(cov_type='cluster', cov_kwds={'groups': ss['ipscntry']})
        m3_ = smf.ols('target_engaged ~ %s*post*impl + C(nace_b) + C(firm_age_ord)' % tcol,
                       data=sd).fit(cov_type='cluster', cov_kwds={'groups': sd['ipscntry']})
    kd = tcol + ':post';  k3 = tcol + ':post:impl'
    cd = md_.params[kd]; sd_ = md_.bse[kd]; pd_ = md_.pvalues[kd]
    c3 = m3_.params[k3]; s3_ = m3_.bse[k3]; p3_ = m3_.pvalues[k3]
    sgd = '***' if pd_<0.01 else '**' if pd_<0.05 else '*' if pd_<0.10 else ''
    sg3 = '***' if p3_<0.01 else '**' if p3_<0.05 else '*' if p3_<0.10 else ''
    print('%-20s %+10.4f %8.4f %6.3f%-3s  %+10.4f %8.4f %6.3f%s' % (
          label, cd, sd_, pd_, sgd, c3, s3_, p3_, sg3))

Treatment group size by definition (df_s)
  Baseline           treat=1:  3226  (15.1%)
  Employees only     treat=1:  1321  (6.2%)
  Turnover only      treat=1:  2662  (12.4%)
  Both (strict)      treat=1:   757  (3.5%)

Treatment def.         DiD coef       SE        p     DDD coef       SE        p
--------------------------------------------------------------------------------
Baseline (any)          +0.0634   0.0289  0.028**      +0.1754   0.0507  0.001***
Employees only          +0.0219   0.0426  0.608        +0.1584   0.0721  0.028**
Turnover only           +0.0793   0.0275  0.004***     +0.1601   0.0396  0.000***
Both (strict)           +0.0587   0.0290  0.043**      +0.1084   0.0975  0.266


#### Below-threshold placebo treatments

These placebos test whether the DiD/DDD effect is **specific to firms at the CSRD
threshold** rather than a general medium-firm trend.

**Placebo A** assigns treatment to firms with \u2265 50 employees **AND** turnover
> \u20ac250k (below the CSRD thresholds of \u2265 250 employees / \u2265 \u20ac10M)
in the full analysis sample.

**Placebo B** applies the same placebo treatment but restricts the sample to firms
that do **not** qualify under the original CSRD criterion (`treat = 0`),
isolating medium firms entirely below the reporting threshold.

A null result in both placebos supports the interpretation that the main estimates
reflect genuine CSRD-threshold effects rather than a broader size trend.

*Requires `employee_50plus` and `turnover_250k` in the parquet.
Re-run `preprocess.py` if these columns are missing.*

In [15]:
# 5.3b  Below-threshold placebo treatments
if 'employee_50plus' not in df.columns or 'turnover_250k' not in df.columns:
    raise RuntimeError('Parquet missing new columns. Run: python preprocess.py')

# ── Placebo treatment: employees >= 50 AND turnover > 250k ──────────────
df_s_pb = df_s.copy()
df_d_pb = df_d.copy()
for df_ in [df_s_pb, df_d_pb]:
    valid = df_['employee_50plus'].notna() & df_['turnover_250k'].notna()
    df_['treat_pb'] = np.where(
        valid,
        ((df_['employee_50plus'] == 1) & (df_['turnover_250k'] == 1)).astype(float),
        np.nan,
    )

df_s_pb = df_s_pb.dropna(subset=['treat_pb'])
df_d_pb = df_d_pb.dropna(subset=['treat_pb'])

# ── Placebo B: restrict to firms below CSRD threshold (treat_original = 0) ──
df_s_pb0 = df_s_pb[df_s_pb['treat'] == 0].copy()
df_d_pb0 = df_d_pb[df_d_pb['treat'] == 0].copy()

# Group sizes
for lbl, ds in [('Placebo A (full sample)', df_s_pb),
                ('Placebo B (treat=0 only)', df_s_pb0)]:
    n   = len(ds)
    n1  = int(ds['treat_pb'].sum())
    pct = 100 * n1 / n
    print('%s:  n=%d,  treat_pb=1: %d (%.1f%%)' % (lbl, n, n1, pct))
print()

# ── DiD and DDD for both placebos ─────────────────────────────────────────
print('%-33s %10s %8s %8s   %10s %8s %8s' % (
      'Placebo', 'DiD coef', 'SE', 'p', 'DDD coef', 'SE', 'p'))
print('-' * 85)
for label, ds, dd in [
    ('A: full (>=50 emp, >250k rev)',   df_s_pb,  df_d_pb),
    ('B: excl. CSRD treated (treat=0)', df_s_pb0, df_d_pb0),
]:
    md_ = smf.ols(
        'target_engaged ~ treat_pb*post + C(nace_b) + C(firm_age_ord)',
        data=ds).fit(cov_type='cluster', cov_kwds={'groups': ds['ipscntry']})
    m3_ = smf.ols(
        'target_engaged ~ treat_pb*post*impl + C(nace_b) + C(firm_age_ord)',
        data=dd).fit(cov_type='cluster', cov_kwds={'groups': dd['ipscntry']})
    cd  = md_.params.get('treat_pb:post',       float('nan'))
    sd_ = md_.bse.get('treat_pb:post',          float('nan'))
    pd_ = md_.pvalues.get('treat_pb:post',      float('nan'))
    c3  = m3_.params.get('treat_pb:post:impl',  float('nan'))
    s3_ = m3_.bse.get('treat_pb:post:impl',     float('nan'))
    p3_ = m3_.pvalues.get('treat_pb:post:impl', float('nan'))
    sgd = '***' if pd_<0.01 else '**' if pd_<0.05 else '*' if pd_<0.10 else ''
    sg3 = '***' if p3_<0.01 else '**' if p3_<0.05 else '*' if p3_<0.10 else ''
    print('%-33s %+10.4f %8.4f %6.3f%-3s  %+10.4f %8.4f %6.3f%s' % (
          label, cd, sd_, pd_, sgd, c3, s3_, p3_, sg3))

print()
print('Main spec reference  — DiD (s2a): treat:post = %.4f  p=%.3f' % (
      s2a.params['treat:post'], s2a.pvalues['treat:post']))
print('Main spec reference  — DDD (s3b): treat:post:impl = %.4f  p=%.3f' % (
      s3b.params['treat:post:impl'], s3b.pvalues['treat:post:impl']))

Placebo A (full sample):  n=21395,  treat_pb=1: 3764 (17.6%)
Placebo B (treat=0 only):  n=18170,  treat_pb=1: 1492 (8.2%)

Placebo                             DiD coef       SE        p     DDD coef       SE        p
-------------------------------------------------------------------------------------
A: full (>=50 emp, >250k rev)        +0.0908   0.0221  0.000***     +0.1661   0.0532  0.002***
B: excl. CSRD treated (treat=0)      +0.0556   0.0246  0.024**      +0.1212   0.0389  0.002***

Main spec reference  — DiD (s2a): treat:post = 0.0634  p=0.028
Main spec reference  — DDD (s3b): treat:post:impl = 0.1754  p=0.001


#### Bandwidth sample: firms just around the CSRD threshold

A composition-effect check: restrict the sample to firms that are
**near the treatment threshold on both size dimensions**.

| Dimension | Bandwidth | Spans threshold at |
|---|---|---|
| Employees | 50\u2013499 | 250 employees |
| Revenue   | \u20ac2 M\u2013\u20ac50 M | \u20ac10 M |

Both flags (`employee_50to499`, `turnover_2m_50m`) must equal 1.
Within this window the original `treat` indicator still applies
(\u2265 250 employees **or** \u2265 \u20ac10 M revenue).

If the treatment effect holds in this narrow bandwidth it is unlikely
to be driven by firm characteristics that differ systematically
between large and small firms far from the threshold.

*Requires `employee_50to499` and `turnover_2m_50m` in the parquet.
Re-run `preprocess.py` if missing.*

In [16]:
# 5.3c  Bandwidth sample around the CSRD threshold
if 'employee_50to499' not in df.columns or 'turnover_2m_50m' not in df.columns:
    raise RuntimeError('Parquet missing bandwidth columns. Run: python preprocess.py')

# ── Build bandwidth sub-samples ─────────────────────────────────────────
bw_mask_s = (df_s['employee_50to499'] == 1) & (df_s['turnover_2m_50m'] == 1)
bw_mask_d = (df_d['employee_50to499'] == 1) & (df_d['turnover_2m_50m'] == 1)
df_s_bw = df_s[bw_mask_s].copy()
df_d_bw = df_d[bw_mask_d].copy()

print('Bandwidth sample (employees 50-499, revenue 2M-50M)')
print('  impl=1 (df_s_bw): n=%-5d  treat=1: %d (%.1f%%)  treat=0: %d (%.1f%%)' % (
      len(df_s_bw),
      (df_s_bw['treat']==1).sum(), 100*(df_s_bw['treat']==1).mean(),
      (df_s_bw['treat']==0).sum(), 100*(df_s_bw['treat']==0).mean()))
print('  all ctry (df_d_bw): n=%-5d  treat=1: %d (%.1f%%)  treat=0: %d (%.1f%%)' % (
      len(df_d_bw),
      (df_d_bw['treat']==1).sum(), 100*(df_d_bw['treat']==1).mean(),
      (df_d_bw['treat']==0).sum(), 100*(df_d_bw['treat']==0).mean()))
print()

# ── OLS / DiD / DDD on bandwidth sample, both outcomes ──────────────────
outcomes_bw = [
    ('target_engaged',    'Climate target', s1a, s2a, s3b, 'treat', 'treat:post', 'treat:post:impl'),
    ('firm_growth_ord_d', 'Firm growth',    s1b, s2b, s3d, 'treat', 'treat:post', 'treat:post:impl'),
]

print('%-16s %-14s %11s %8s %8s   %11s %8s %8s' % (
      'Outcome', 'Spec', 'BW coef', 'SE', 'p', 'Full coef', 'SE', 'p'))
print('-' * 88)
for (col, lbl, m_ols_f, m_did_f, m_ddd_f, k_ols, k_did, k_ddd) in outcomes_bw:
    m_bw_ols = smf.ols('%s ~ treat + C(nace_b) + C(firm_age_ord)' % col,
                        data=df_s_bw).fit(
                            cov_type='cluster', cov_kwds={'groups': df_s_bw['ipscntry']})
    m_bw_did = smf.ols('%s ~ treat*post + C(nace_b) + C(firm_age_ord)' % col,
                        data=df_s_bw).fit(
                            cov_type='cluster', cov_kwds={'groups': df_s_bw['ipscntry']})
    m_bw_ddd = smf.ols('%s ~ treat*post*impl + C(nace_b) + C(firm_age_ord)' % col,
                        data=df_d_bw).fit(
                            cov_type='cluster', cov_kwds={'groups': df_d_bw['ipscntry']})
    for spec, mbw, mf, key in [('OLS', m_bw_ols, m_ols_f, k_ols),
                                ('DiD', m_bw_did, m_did_f, k_did),
                                ('DDD', m_bw_ddd, m_ddd_f, k_ddd)]:
        c_bw = mbw.params[key];  s_bw = mbw.bse[key];  p_bw = mbw.pvalues[key]
        c_f  = mf.params[key];   s_f  = mf.bse[key];   p_f  = mf.pvalues[key]
        sg   = '***' if p_bw<0.01 else '**' if p_bw<0.05 else '*' if p_bw<0.10 else ''
        out_lbl = lbl if spec == 'OLS' else ''
        print('%-16s %-14s %+11.4f %8.4f %6.3f%-3s  %+11.4f %8.4f %6.3f' % (
              out_lbl, spec, c_bw, s_bw, p_bw, sg, c_f, s_f, p_f))
    print()
print('BW = bandwidth sample (n_s=%d, n_d=%d). Full = full sample.' % (
      len(df_s_bw), len(df_d_bw)))
print('SE clustered by ipscntry in both.')

Bandwidth sample (employees 50-499, revenue 2M-50M)
  impl=1 (df_s_bw): n=2384   treat=1: 1319 (55.3%)  treat=0: 1065 (44.7%)
  all ctry (df_d_bw): n=2909   treat=1: 1627 (55.9%)  treat=0: 1282 (44.1%)

Outcome          Spec               BW coef       SE        p     Full coef       SE        p
----------------------------------------------------------------------------------------
Climate target   OLS                +0.1295   0.0264  0.000***      +0.2608   0.0323  0.000
                 DiD                +0.0544   0.0484  0.261         +0.0634   0.0289  0.028
                 DDD                +0.1081   0.0633  0.088*        +0.1754   0.0507  0.001

Firm growth      OLS                +0.0804   0.0239  0.001***      +0.2302   0.0252  0.000
                 DiD                -0.1856   0.0425  0.000***      +0.0649   0.0265  0.014
                 DDD                -0.2027   0.0933  0.030**       -0.0524   0.0664  0.430

BW = bandwidth sample (n_s=2384, n_d=2909). Full = full samp

### 5.4  DiD and DDD with Alternative Outcome Variables

Running DiD **and** DDD on pre-determined firm characteristics (`firm_age_ord`)
serves as a **falsification test**: a significant coefficient would indicate
compositional shifts rather than genuine treatment effects.
For forward-looking variables (`green_offer_ord_d`, `green_staff_any`, etc.) the
estimates capture potential policy spillovers or mechanisms.

In [17]:
# 5.4  DiD and DDD on alternative outcomes
alt_out = [
    ('firm_age_ord',            'Firm age ordinal [falsification]'),
    ('green_staff_any',         'Green staff (any)'),
    ('investment_dummy',        'Investment dummy'),
    ('barrier_institutional_d', 'Institutional barriers'),
    ('support_knowledge_d',     'Knowledge support'),
    ('green_offer_ord_d',       'Green offer (binary)'),
]

print('%-40s %10s %8s %8s   %10s %8s %8s' % (
      'Outcome', 'DiD coef', 'SE', 'p', 'DDD coef', 'SE', 'p'))
print('-' * 90)
for col, label in alt_out:
    # DiD (impl=1 sample)
    sub_s = df_s.dropna(subset=[col]).copy()
    rhs_s = ('treat*post + C(nace_b)' if col == 'firm_age_ord'
             else 'treat*post + C(nace_b) + C(firm_age_ord)')
    m_did = smf.ols('%s ~ %s' % (col, rhs_s), data=sub_s).fit(
                cov_type='cluster', cov_kwds={'groups': sub_s['ipscntry']})
    # DDD (all countries)
    sub_d = df_d.dropna(subset=[col]).copy()
    rhs_d = ('treat*post*impl + C(nace_b)' if col == 'firm_age_ord'
             else 'treat*post*impl + C(nace_b) + C(firm_age_ord)')
    m_ddd = smf.ols('%s ~ %s' % (col, rhs_d), data=sub_d).fit(
                cov_type='cluster', cov_kwds={'groups': sub_d['ipscntry']})
    cd  = m_did.params.get('treat:post',      float('nan'))
    sd  = m_did.bse.get('treat:post',         float('nan'))
    pd_ = m_did.pvalues.get('treat:post',     float('nan'))
    c3  = m_ddd.params.get('treat:post:impl', float('nan'))
    s3  = m_ddd.bse.get('treat:post:impl',    float('nan'))
    p3  = m_ddd.pvalues.get('treat:post:impl',float('nan'))
    sgd = '***' if pd_<0.01 else '**' if pd_<0.05 else '*' if pd_<0.10 else ''
    sg3 = '***' if p3<0.01  else '**' if p3<0.05  else '*' if p3<0.10  else ''
    print('%-40s %+10.4f %8.4f %6.3f%-3s  %+10.4f %8.4f %6.3f%s' % (
          label, cd, sd, pd_, sgd, c3, s3, p3, sg3))
print()
print('DiD: treat*post + sector FEs + firm_age FE (omitted when outcome=firm_age_ord).')
print('DDD: treat*post*impl + same FEs. SE clustered by ipscntry.')

Outcome                                    DiD coef       SE        p     DDD coef       SE        p
------------------------------------------------------------------------------------------
Firm age ordinal [falsification]            -0.0510   0.0327  0.119        +0.0931   0.1177  0.429
Green staff (any)                           +0.0472   0.0243  0.052*       +0.1495   0.0431  0.001***
Investment dummy                            -0.0034   0.0148  0.818        +0.0535   0.0274  0.051*
Institutional barriers                      +0.0526   0.0260  0.043**      +0.0865   0.0722  0.231
Knowledge support                           +0.0304   0.0199  0.127        +0.0805   0.0533  0.131
Green offer (binary)                        +0.0352   0.0229  0.125        +0.0606   0.0379  0.110

DiD: treat*post + sector FEs + firm_age FE (omitted when outcome=firm_age_ord).
DDD: treat*post*impl + same FEs. SE clustered by ipscntry.


### 5.5  OLS / DiD / DDD with East\u2013West Control

`east_west = 1` for Western EU member states, `0` for Central/Eastern EU.
Adding this country-level dummy controls for unobserved geographic differences
in baseline climate engagement.  In DDD models `east_west` is collinear with
country-level variation in `impl`, so coefficient SEs may be inflated; the
direction and significance of `treat:post:impl` provides the relevant comparison.

| New model | Base | Added |
|---|---|---|
| s1a\_ew / s1b\_ew | s1a / s1b | + `east_west` (OLS) |
| s2a\_ew / s2b\_ew | s2a / s2b | + `east_west` (DiD) |
| s3b\_ew / s3d\_ew | s3b / s3d | + `east_west` (DDD) |

In [18]:
# 5.5  East-West robustness
s1a_ew = smf.ols('target_engaged    ~ treat + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s1b_ew = smf.ols('firm_growth_ord_d ~ treat + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s2a_ew = smf.ols('target_engaged    ~ treat*post + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s2b_ew = smf.ols('firm_growth_ord_d ~ treat*post + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s3b_ew = smf.ols('target_engaged    ~ treat*post*impl + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_d).fit(**cl(df_d))
s3d_ew = smf.ols('firm_growth_ord_d ~ treat*post*impl + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_d).fit(**cl(df_d))

ew_specs = [
    ('OLS 1a', s1a_ew, s1a, 'treat'),
    ('OLS 1b', s1b_ew, s1b, 'treat'),
    ('DiD 2a', s2a_ew, s2a, 'treat:post'),
    ('DiD 2b', s2b_ew, s2b, 'treat:post'),
    ('DDD 3b', s3b_ew, s3b, 'treat:post:impl'),
    ('DDD 3d', s3d_ew, s3d, 'treat:post:impl'),
]

print('%-10s %11s %8s %6s   %11s %8s %6s   %9s %8s' % (
      'Model', '+EW coef', 'SE', 'p', 'Base coef', 'SE', 'p', 'EW coef', 'EW SE'))
print('-' * 88)
for lbl, mew, mbase, key in ew_specs:
    c_ew = mew.params[key];   s_ew = mew.bse[key];   p_ew = mew.pvalues[key]
    c_b  = mbase.params[key]; s_b  = mbase.bse[key]; p_b  = mbase.pvalues[key]
    c_e  = mew.params.get('east_west', float('nan'))
    s_e  = mew.bse.get('east_west',    float('nan'))
    sg   = '***' if p_ew<0.01 else '**' if p_ew<0.05 else '*' if p_ew<0.10 else ''
    print('%-10s %+11.4f %8.4f %4.3f%-3s  %+11.4f %8.4f %4.3f   %+9.4f %8.4f' % (
          lbl, c_ew, s_ew, p_ew, sg, c_b, s_b, p_b, c_e, s_e))

Model         +EW coef       SE      p     Base coef       SE      p     EW coef    EW SE
----------------------------------------------------------------------------------------
OLS 1a         +0.2321   0.0289 0.000***      +0.2608   0.0323 0.000     +0.1452   0.0349
OLS 1b         +0.1713   0.0130 0.000***      +0.2302   0.0252 0.000     -0.0089   0.0408
DiD 2a         +0.0545   0.0280 0.051*        +0.0634   0.0289 0.028     +0.1448   0.0351
DiD 2b         -0.1057   0.0381 0.006***      +0.0649   0.0265 0.014     -0.0079   0.0407
DDD 3b         +0.1663   0.0502 0.001***      +0.1754   0.0507 0.001     +0.1446   0.0349
DDD 3d         -0.0519   0.0663 0.434         -0.0524   0.0664 0.430     -0.0083   0.0404


---
## 6. Extension — Resource-Efficiency Domain Outcomes

The main analysis uses `target_engaged` and `firm_growth_ord_d` as outcomes.
Here we re-run the three core specifications — **OLS**, **DiD**, and **DDD** —
using each of the ten resource-efficiency *action-maturity* indicators as the
outcome variable.  Each `{domain}_maturity_d` equals 1 if the firm currently
undertakes the action **or** plans to (built from the `q1`/`q2` pairs in
`preprocess.py`), and 0 otherwise.

| Domain | Variable |
|---|---|
| Water efficiency        | `water_maturity_d` |
| Energy saving           | `energy_saving_maturity_d` |
| Green energy            | `green_energy_maturity_d` |
| Materials efficiency    | `materials_maturity_d` |
| Green suppliers         | `green_suppliers_maturity_d` |
| Waste reduction         | `waste_reduction_maturity_d` |
| Selling residues        | `sell_residues_maturity_d` |
| Recycling               | `recycling_maturity_d` |
| Eco-design              | `eco_design_maturity_d` |
| Other                   | `other_maturity_d` |

All outcomes are binary, so OLS is a linear probability model (consistent with
Sections 1–3).  Controls: sector FEs + firm-age FEs.  SE clustered by country.

In [ ]:
# 6.  OLS / DiD / DDD for the resource-efficiency domain outcomes
domain_outcomes = [
    ('water_maturity_d',           'Water efficiency'),
    ('energy_saving_maturity_d',   'Energy saving'),
    ('green_energy_maturity_d',    'Green energy'),
    ('materials_maturity_d',       'Materials efficiency'),
    ('green_suppliers_maturity_d', 'Green suppliers'),
    ('waste_reduction_maturity_d', 'Waste reduction'),
    ('sell_residues_maturity_d',   'Selling residues'),
    ('recycling_maturity_d',       'Recycling'),
    ('eco_design_maturity_d',      'Eco-design'),
    ('other_maturity_d',           'Other'),
]

def _st(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

print('%-22s %9s %7s %7s   %9s %7s %7s   %9s %7s %7s' % (
      'Domain', 'OLS', 'SE', 'p', 'DiD', 'SE', 'p', 'DDD', 'SE', 'p'))
print('%-22s %9s %7s %7s   %9s %7s %7s   %9s %7s %7s' % (
      '', 'treat', '', '', 'tr:post', '', '', 'tr:po:impl', '', ''))
print('-' * 96)

domain_rows = []
for col, label in domain_outcomes:
    sub_s = df_s.dropna(subset=[col]).copy()
    sub_d = df_d.dropna(subset=[col]).copy()
    m_ols = smf.ols('%s ~ treat + C(nace_b) + C(firm_age_ord)' % col,
                    data=sub_s).fit(cov_type='cluster', cov_kwds={'groups': sub_s['ipscntry']})
    m_did = smf.ols('%s ~ treat*post + C(nace_b) + C(firm_age_ord)' % col,
                    data=sub_s).fit(cov_type='cluster', cov_kwds={'groups': sub_s['ipscntry']})
    m_ddd = smf.ols('%s ~ treat*post*impl + C(nace_b) + C(firm_age_ord)' % col,
                    data=sub_d).fit(cov_type='cluster', cov_kwds={'groups': sub_d['ipscntry']})
    co, so, po = m_ols.params['treat'],            m_ols.bse['treat'],            m_ols.pvalues['treat']
    cd, sd, pd_ = m_did.params['treat:post'],       m_did.bse['treat:post'],       m_did.pvalues['treat:post']
    c3, s3, p3 = m_ddd.params['treat:post:impl'],   m_ddd.bse['treat:post:impl'],  m_ddd.pvalues['treat:post:impl']
    print('%-22s %+9.4f %7.4f %5.3f%-2s %+9.4f %7.4f %5.3f%-2s %+9.4f %7.4f %5.3f%-2s' % (
          label, co, so, po, _st(po), cd, sd, pd_, _st(pd_), c3, s3, p3, _st(p3)))
    domain_rows.append({
        'domain': col, 'label': label,
        'ols_treat': co, 'ols_se': so, 'ols_p': po,
        'did_treatpost': cd, 'did_se': sd, 'did_p': pd_,
        'ddd_treatpostimpl': c3, 'ddd_se': s3, 'ddd_p': p3,
    })

print('-' * 96)
print('OLS/DiD on impl=1 sample (df_s); DDD on full sample (df_d).')
print('Controls: sector FEs + firm-age FEs. SE clustered by ipscntry.')
print('Stars: *** p<0.01, ** p<0.05, * p<0.10.')

domain_table = pd.DataFrame(domain_rows)
domain_table.to_csv('domain_results.csv', index=False)
print('\nSaved: domain_results.csv')